## Import Libraries

In [ ]:
# imports
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

print("Pandas version:", pd.__version__)
print("Numpy version:", np.__version__)

## Load Dataset and Make Inspection

In [2]:

# Load raw dataset
df = pd.read_csv("../data/raw/synthetic_loan_approval_dataset.csv")

# Inspect 
print(df.head(5))
df.info()


  Application_ID  Age  Gender Marital_Status      Education  Dependents  \
0      APP100000   24  Female         Single       Graduate           0   
1      APP100001   26    Male        Married       Graduate           2   
2      APP100002   58  Female        Married  Post Graduate           1   
3      APP100003   38  Others        Married       Graduate           1   
4      APP100004   32  Female         Single            PhD           0   

  Residence_Type  City_Tier Employment_Type  Years_at_Current_Job  ...  \
0          Rural          3         Private                     1  ...   
1          Urban          1         Private                     0  ...   
2          Urban          1      Government                    21  ...   
3          Rural          4         Private                     2  ...   
4          Rural          4      Government                     1  ...   

   Aadhaar_Verified  Loan_Purpose  Loan_Amount  Loan_Tenure  Interest_Rate  \
0               Yes       

## Feature Selection & Dropping Redundant Columns



In this step, we drop specific columns from the dataset to refine our feature set for the decision tree model:

* **`Application_ID`:** Unique identifier; contains no predictive signal.
* **`Gender`:** Dropped to prevent demographic bias and maintain ethical lending standards.
* **`Annual_Income`:** Excluded because we have monthly income already which can be used to compute annual, no new information provided by it.
* **`PAN_Verified` & `Aadhaar_Verified`:** Verification flags that do not directly measure credit risk.
* **`Property_Value`:** Redundant, as collateral risk is already captured by `Loan_Amount` and `Loan_to_Value` (LTV).

In [3]:

# Drop unwanted columns
columns_to_drop = [
    "Application_ID",
    "Gender",
    "Annual_Income",
    "PAN_Verified",
    "Aadhaar_Verified",
    "Property_Value"
]

df.drop(columns=columns_to_drop, inplace=True)

In [4]:
# verifying dropped columns
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 33 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Age                      20000 non-null  int64  
 1   Marital_Status           20000 non-null  str    
 2   Education                20000 non-null  str    
 3   Dependents               20000 non-null  int64  
 4   Residence_Type           20000 non-null  str    
 5   City_Tier                20000 non-null  int64  
 6   Employment_Type          20000 non-null  str    
 7   Years_at_Current_Job     20000 non-null  int64  
 8   Total_Work_Experience    20000 non-null  int64  
 9   Monthly_Income           20000 non-null  int64  
 10  Other_Income             19587 non-null  float64
 11  Existing_Loans           20000 non-null  int64  
 12  Existing_Loan_Amount     20000 non-null  int64  
 13  Monthly_EMI              20000 non-null  int64  
 14  Debt_to_Income           20000 no

## Define Features ($X$) and Target ($y$)



Before splitting the dataset for model training, we separate our dataset into the feature matrix ($X$) and the target vector ($y$).

We separate the dataset into X and y to tell the machine learning algorithm which data are the input features (X) and which data are the target/label (y) that we want the model to predict.

* **Feature Matrix ($X$):** Contains all predictor variables (both raw numerical metrics and engineered features) while excluding the target column.
* **Target Vector ($y$):** Contains the binary ground-truth labels (`Loan_Status`), where `1` represents an approved loan and `0` represents a rejected loan.

In [5]:
# Define features (X) and target (y)

X = df.drop("Loan_Status", axis=1)
y = df["Loan_Status"]

# Verify the results
print("=========================================")
print("X AND y VERIFICATION")
print("=========================================")

print("\nShape of X:", X.shape)
print("Shape of y:", y.shape)

print("\nFeatures in X:")
print(X.columns.tolist())

print("\nTarget column:")
print(y.name)

X AND y VERIFICATION

Shape of X: (20000, 32)
Shape of y: (20000,)

Features in X:
['Age', 'Marital_Status', 'Education', 'Dependents', 'Residence_Type', 'City_Tier', 'Employment_Type', 'Years_at_Current_Job', 'Total_Work_Experience', 'Monthly_Income', 'Other_Income', 'Existing_Loans', 'Existing_Loan_Amount', 'Monthly_EMI', 'Debt_to_Income', 'Savings', 'Investments', 'Bank_Balance', 'Credit_Card_Utilization', 'Number_of_Bank_Accounts', 'Number_of_Credit_Cards', 'Credit_Score', 'Loan_Defaults', 'Missed_Payments', 'Tax_Return_Filed', 'Loan_Purpose', 'Loan_Amount', 'Loan_Tenure', 'Interest_Rate', 'Collateral', 'Collateral_Value', 'Loan_to_Value']

Target column:
Loan_Status


### 📊 Dataset Separation Verification

The features and target variable have been successfully partitioned:

* **Predictor Matrix ($X$):** Consists of $43$ input features across $20,000$ samples.
* **Target Variable ($y$):** Consists of $20,000$ binary labels corresponding to `Loan_Status`.

> **Next Step:** With $X$ and $y$ cleanly defined, we are ready to perform a stratified train-test split to evaluate our model's performance on unseen data.

## Train/Test Split

### ✂️ Train-Test Partitioning

We split our feature matrix ($X$) and target vector ($y$) into training and testing subsets to evaluate how well our model generalizes to unseen data.

* **Split Ratio ($80/20$):** $80\%$ of the data is allocated for model training, leaving $20\%$ as an independent test holdout.


In [6]:
from sklearn.model_selection import train_test_split

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Verify the split
print("=========================================")
print("TRAIN / TEST SPLIT VERIFICATION")
print("=========================================")

print("\nX_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

print("\ny_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

print("\nTraining percentage:",
      round(len(X_train) / len(X) * 100, 2), "%")

print("Testing percentage:",
      round(len(X_test) / len(X) * 100, 2), "%")

TRAIN / TEST SPLIT VERIFICATION

X_train shape: (16000, 32)
X_test shape: (4000, 32)

y_train shape: (16000,)
y_test shape: (4000,)

Training percentage: 80.0 %
Testing percentage: 20.0 %


---
### Data Cleaning: Missing Value Imputation

Before training our model, we need to address any missing (`NA`) entries in our feature set:

1. **Inspection:** Identify columns containing missing values and their respective proportions.
2. **Strategy:** Apply specific handling techniques (e.g., median imputation for continuous features like `LTV`, or dropping rows where critical targets are missing).
3. **Verification:** Confirm that no null entries remain in the preprocessed DataFrame.
---

In [7]:
# Inspecting for total number of missing values in the dataset
missing_values = df.isnull().sum()
print("\nMissing values in each column:\n", missing_values)


Missing values in each column:
 Age                           0
Marital_Status                0
Education                     0
Dependents                    0
Residence_Type                0
City_Tier                     0
Employment_Type               0
Years_at_Current_Job          0
Total_Work_Experience         0
Monthly_Income                0
Other_Income                413
Existing_Loans                0
Existing_Loan_Amount          0
Monthly_EMI                   0
Debt_to_Income                0
Savings                       0
Investments                 409
Bank_Balance                193
Credit_Card_Utilization       0
Number_of_Bank_Accounts       0
Number_of_Credit_Cards        0
Credit_Score                118
Loan_Defaults                 0
Missed_Payments               0
Tax_Return_Filed            175
Loan_Purpose                  0
Loan_Amount                   0
Loan_Tenure                   0
Interest_Rate               118
Collateral                    0
Collate

### Summary of Identified Missing (`NA`) Values

A total of 8 feature columns contain missing values out of **20,000 total records**:

| Feature Column | Missing Value Count (`NA`) | Percentage of Total Data (%) |
| :--- | :--- | :--- |
| **`Loan_to_Value`** | 7,476 | **37.38%** |
| **`Collateral_Value`** | 7,395 | **36.98%** |
| **`Other_Income`** | 413 | **2.07%** |
| **`Investments`** | 409 | **2.05%** |
| **`Bank_Balance`** | 193 | **0.97%** |
| **`Tax_Return_Filed`** | 175 | **0.88%** |
| **`Credit_Score`** | 118 | **0.59%** |
| **`Interest_Rate`** | 118 | **0.59%** |

 **Key Observation:** `Loan_to_Value` and `Collateral_Value` make up the vast majority of missing entries (over **37%** of the dataset). Because these are correlated with `Loan_Amount`, they will require systematic calculation or targeted imputation before training the model.

### Missing Value Handling Strategy

Based on domain context, feature types, and the proportion of missing entries, the following imputation and reconstruction strategies will be applied:

| Feature Column | Feature Type | Missing (`NA`) % | Imputation / Reconstruction Strategy | Justification |
| :--- | :--- | :--- | :--- | :--- |
| **`Loan_to_Value` (LTV)** | Numerical | 37.38% | **Mathematical Reconstruction / Median** | Recalculate using $\text{LTV} = \frac{\text{Loan Amount}}{\text{Collateral Value}} \times 100$ where collateral exists; if collateral is also missing then impute the LVT with median . |
| **`Collateral_Value`** | Numerical | 36.98% | **Derive via LTV / Zero-Fill** | Back-calculate using $\text{Collateral} = \frac{\text{Loan Amount}}{\text{LTV}}$ where LTV is present; fill unsecured loans with `0`. |
| **`Other_Income`** | Numerical | 2.07% | **Zero-Fill (`0`)** | Missing value implies applicant has no secondary source of income. |
| **`Investments`** | Numerical | 2.05% | **Zero-Fill (`0`)** | Missing value implies applicant holds no external financial investments/assets. |
| **`Bank_Balance`** | Numerical | 0.97% | **Median Imputation** | Skewed financial metric; filling with column median avoids outlier distortion. |
| **`Tax_Return_Filed`** | Categorical | 0.88% | **Mode / Constant Flag (`"Unknown"`)** | Fill missing values with the most frequent category or explicitly flag as `"Unknown"`. |
| **`Credit_Score`** | Numerical | 0.59% | **Grouped Median / Overall Median** | Minimal missingness (<1%); impute using overall median credit score or median grouped by loan status. |
| **`Interest_Rate`** | Numerical | 0.59% | **Median / Subgroup Median** | Minimal missingness (<1%); impute using median interest rate based on credit score/loan tier. |

---

### Missing Data: Feature 01 — Evaluating `Loan_to_Value`

In [8]:
print("=== Loan_to_Value Distribution ===")
print(df['Loan_to_Value'].describe())

print("\n=== Loan_to_Value Quantiles ===")
print(df['Loan_to_Value'].quantile([0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]))

print("\n=== Loan_to_Value Missingness ===")
print(df['Loan_to_Value'].isna().sum())
print("Percentage missing:", df['Loan_to_Value'].isna().mean() * 100)



=== Loan_to_Value Distribution ===
count    12524.000000
mean        67.701789
std         12.733992
min         45.480000
25%         57.230000
50%         65.780000
75%         77.180000
max         97.941706
Name: Loan_to_Value, dtype: float64

=== Loan_to_Value Quantiles ===
0.01    47.3992
0.05    50.0615
0.25    57.2300
0.50    65.7800
0.75    77.1800
0.95    91.4200
0.99    95.0000
Name: Loan_to_Value, dtype: float64

=== Loan_to_Value Missingness ===
7476
Percentage missing: 37.38


### Statistical Analysis: Why Median Imputation for `Loan_to_Value`?

By inspecting the summary statistics and quantile distributions for **`Loan_to_Value` (LTV)**, we observe key distribution traits that dictate our imputation strategy:

#### Key Statistical Findings:
* **Right-Skewed Distribution:** The mean ($\approx 67.70\%$) is higher than the median / 50th percentile ($\approx 65.78\%$).
* **Outlier / Tail Impact:** The top percentile reaches **97.94%**, pulling the mean upward due to high-risk upper-bound loans.
* **Stable Central Range:** The Interquartile Range (IQR) spans **57.23% to 77.18%**, showing that typical loan ratios cluster predictably around 65.78%.

> **Conclusion & Decision:**  
> Because **37.38% (7,476 rows)** of the data is missing, using the **Mean** would systematically overestimate risk by shifting imputed values toward high extreme ratios. The **Median (65.78%)** represents the true 50th percentile and is robust against tail-end skewness.

### Handle Missing Values in `Loan_to_Value`
**Strategy:** Median Imputation

In [9]:

# Calculate the median of the available Loan_to_Value values
ltv_median = df['Loan_to_Value'].median()

# Replace all missing Loan_to_Value values with the median
df['Loan_to_Value'] = df['Loan_to_Value'].fillna(ltv_median)

print("Loan_to_Value missing values have been imputed.")
print("Median value used:", ltv_median)

Loan_to_Value missing values have been imputed.
Median value used: 65.78


### Verify `Loan_to_Value` Imputation

In [10]:

# Count the remaining missing values
remaining_missing = df['Loan_to_Value'].isna().sum()

print("Remaining missing Loan_to_Value values:", remaining_missing)

# Confirm whether the imputation was successful
if remaining_missing == 0:
    print("Verification successful: No missing values remain in Loan_to_Value.")
else:
    print("Verification failed: Missing values still remain in Loan_to_Value.")

Remaining missing Loan_to_Value values: 0
Verification successful: No missing values remain in Loan_to_Value.


### Missing Data: Feature 02 — Evaluating `Collateral_Value`

In [11]:
print("=== Collateral_Value Missingness ===")

# Count missing values
missing_count = df['Collateral_Value'].isna().sum()
# Calculate percentage missing
missing_percentage = df['Collateral_Value'].isna().mean() * 100

print("Number of missing values:", missing_count)
print("Percentage missing:", round(missing_percentage, 2), "%")


print("\n=== Collateral_Value Distribution ===")

# Display descriptive statistics for available values
print(df['Collateral_Value'].describe())


print("\n=== Collateral_Value Quantiles ===")

# Examine the distribution at different percentiles
print(
    df['Collateral_Value'].quantile(
        [0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
    )
)

=== Collateral_Value Missingness ===
Number of missing values: 7395
Percentage missing: 36.98 %

=== Collateral_Value Distribution ===
count    1.260500e+04
mean     4.033661e+06
std      3.971447e+06
min      4.754000e+03
25%      1.279655e+06
50%      3.151921e+06
75%      5.716036e+06
max      1.220902e+08
Name: Collateral_Value, dtype: float64

=== Collateral_Value Quantiles ===
0.01      167099.32
0.05      397173.40
0.25     1279655.00
0.50     3151921.00
0.75     5716036.00
0.95    10483958.40
0.99    15265989.24
Name: Collateral_Value, dtype: float64


### Missing Data Assessment: `Collateral_Value`

Evaluating central metrics to determine the safest imputation method for **7,395 missing rows (36.98%)**:

| Metric | Formatted Value | Impact on Imputation Choice |
| :--- | :--- | :--- |
| **Mean** | `4,033,661` | **Heavily Inflated:** Pulled upward by high-end outliers (max: 122M). |
| **Median (50%)** | `3,151,921` | **Robust Baseline:** Unaffected by upper-tail extremes. |
| **IQR (25% – 75%)** | `1,279,655 – 5,716,036` | Demonstrates typical middle-50% collateral spread. |

**Decision:** We choose **Median Imputation (`3,151,921`)** to prevent high-value properties from artificially inflating credit collateral estimates across missing records.

### Handle Missing Values in `Collateral_Value`
**Strategy:** Median Imputation

In [12]:
# Calculate the median of the available Collateral_Value values
collateral_median = df['Collateral_Value'].median()

# Replace all missing Collateral_Value values with the median
df['Collateral_Value'] = df['Collateral_Value'].fillna(collateral_median)

print("Collateral_Value missing values have been imputed.")
print("Median value used:", collateral_median)

Collateral_Value missing values have been imputed.
Median value used: 3151921.0


### Verify `Collateral_Value` Imputation

In [13]:
# Count the remaining missing values
remaining_missing = df['Collateral_Value'].isna().sum()

print("Remaining missing Collateral_Value values:", remaining_missing)

# Confirm whether the imputation was successful
if remaining_missing == 0:
    print("Verification successful: No missing values remain in Collateral_Value.")
else:
    print("Verification failed: Missing values still remain in Collateral_Value.")

Remaining missing Collateral_Value values: 0
Verification successful: No missing values remain in Collateral_Value.


### Handle Remaining Missing Values



The remaining six features are imputed using domain logic (zero-fill, explicit category) or robust central tendencies (median):

| Feature | Missing Strategy | Rationale |
| :--- | :--- | :--- |
| **`Other_Income`** | Zero-Fill (`0`) | Absence of recorded value indicates no secondary income stream. |
| **`Investments`** | Zero-Fill (`0`) | Absence of recorded value indicates no reported investment holdings. |
| **`Bank_Balance`** | Median Imputation | Robust to skewness caused by high-balance account outliers. |
| **`Tax_Return_Filed`** | Constant (`'Unknown'`) | Preserves missingness explicitly as a distinct categorical flag. |
| **`Credit_Score`** | Median Imputation | Protects against tail-end credit extremes. |
| **`Interest_Rate`** | Median Imputation | Provides a stable central rate un-skewed by subprime interest outliers. |

In [14]:

# 1. Other_Income
# Missing values are assumed to indicate no other income.
df['Other_Income'] = df['Other_Income'].fillna(0)

# 2. Investments
# Missing values are assumed to indicate no reported investments.
df['Investments'] = df['Investments'].fillna(0)

# 3. Bank_Balance
# Use the median because financial balances can be skewed by large values.
bank_balance_median = df['Bank_Balance'].median()
df['Bank_Balance'] = df['Bank_Balance'].fillna(bank_balance_median)

# 4. Tax_Return_Filed
# Use "Unknown" to preserve the fact that the original value was missing.
df['Tax_Return_Filed'] = df['Tax_Return_Filed'].fillna('Unknown')

# 5. Credit_Score
# Use the median because it is robust to potential extreme values.
credit_score_median = df['Credit_Score'].median()
df['Credit_Score'] = df['Credit_Score'].fillna(credit_score_median)

# 6. Interest_Rate
# Use the median because it is robust to potential extreme values.
interest_rate_median = df['Interest_Rate'].median()
df['Interest_Rate'] = df['Interest_Rate'].fillna(interest_rate_median)

print("Missing values in the six remaining features have been imputed.")

Missing values in the six remaining features have been imputed.


### Verify Remaining Missing Values

In [15]:

remaining_features = [
    'Other_Income',
    'Investments',
    'Bank_Balance',
    'Tax_Return_Filed',
    'Credit_Score',
    'Interest_Rate'
]

# Count remaining missing values
remaining_missing = df[remaining_features].isna().sum()

print("=== Remaining Missing Values ===")
print(remaining_missing)

# Check whether all missing values have been handled
if remaining_missing.sum() == 0:
    print("\nVerification successful: No missing values remain in these features.")
else:
    print("\nVerification failed: Some missing values still remain.")

=== Remaining Missing Values ===
Other_Income        0
Investments         0
Bank_Balance        0
Tax_Return_Filed    0
Credit_Score        0
Interest_Rate       0
dtype: int64

Verification successful: No missing values remain in these features.


## Identify Numerical Features for Outlier Analysis

In [16]:


numerical_columns = df.select_dtypes(include=['number']).columns.tolist()

print("=== Numerical Features ===")
print(numerical_columns)

print("\nNumber of numerical features:", len(numerical_columns))

=== Numerical Features ===
['Age', 'Dependents', 'City_Tier', 'Years_at_Current_Job', 'Total_Work_Experience', 'Monthly_Income', 'Other_Income', 'Existing_Loans', 'Existing_Loan_Amount', 'Monthly_EMI', 'Debt_to_Income', 'Savings', 'Investments', 'Bank_Balance', 'Credit_Card_Utilization', 'Number_of_Bank_Accounts', 'Number_of_Credit_Cards', 'Credit_Score', 'Loan_Defaults', 'Missed_Payments', 'Loan_Amount', 'Loan_Tenure', 'Interest_Rate', 'Collateral_Value', 'Loan_to_Value']

Number of numerical features: 25


### Outlier Analysis: Interquartile Range (IQR)

The **Interquartile Range (IQR)** is a statistical method used to identify potential outliers in numerical data. It is calculated as the difference between the third quartile ($Q_3$) and the first quartile ($Q_1$):

$$\text{IQR} = Q_3 - Q_1$$

In this study, a value is considered an outlier if it meets either condition:

* **Lower Outlier Boundary:** $Q_1 - 1.5(\text{IQR})$
* **Upper Outlier Boundary:** $Q_3 + 1.5(\text{IQR})$

This method will be applied to the numerical features in the dataset to identify unusually high or low values that may affect the performance of the machine learning model.

### Outlier Detection Example using IQR

Given the dataset:
`10, 12, 13, 14, 15, 16, 17, 18, 20, 50`

Visually, **50** appears unusually high. We can use the Interquartile Range (IQR) method to mathematically verify if it is an outlier.

---

#### Step 1: Find $Q_1$ and $Q_3$

1. **Find the Median ($Q_2$):**  
   With $N = 10$ values, the median lies between the 5th and 6th positions:
   $$Q_2 = \frac{15 + 16}{2} = 15.5$$

2. **Split the Data:**
   * **Lower Half:** `10, 12, 13, 14, 15`
   * **Upper Half:** `16, 17, 18, 20, 50`

3. **Calculate Quartiles & IQR:**
   * $Q_1 = 13$ (median of lower half)
   * $Q_3 = 18$ (median of upper half)
   * $\text{IQR} = Q_3 - Q_1 = 18 - 13 = 5$

---

#### Step 2: Calculate the Outlier Boundaries

* **Lower Boundary:** $Q_1 - 1.5(\text{IQR}) = 13 - 1.5(5) = 5.5$
* **Upper Boundary:** $Q_3 + 1.5(\text{IQR}) = 18 + 1.5(5) = 25.5$

Acceptable values fall within **5.5 to 25.5**.

---

#### Step 3: Identify Outliers

Evaluating our values against the boundaries $[5.5, 25.5]$:

* All values from 10 to 20 fall within bounds.
* **$50 > 25.5$** $\rightarrow$ **50 is an outlier.**

### DETECT POTENTIAL OUTLIERS IN NUMERICAL FEATURES

In [17]:


# Create a list to store outlier results
outlier_results = []

# Calculate IQR-based outliers for each numerical feature
for column in numerical_columns:
    
    # Calculate Q1 and Q3
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    
    # Calculate the Interquartile Range
    IQR = Q3 - Q1
    
    # Calculate lower and upper bounds
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Count potential outliers
    outlier_count = (
        (df[column] < lower_bound) |
        (df[column] > upper_bound)
    ).sum()
    
    # Calculate percentage of potential outliers
    outlier_percentage = (outlier_count / len(df)) * 100
    
    # Store the results
    outlier_results.append({
        'Feature': column,
        'Q1': Q1,
        'Q3': Q3,
        'IQR': IQR,
        'Lower_Bound': lower_bound,
        'Upper_Bound': upper_bound,
        'Outlier_Count': outlier_count,
        'Outlier_Percentage': outlier_percentage
    })

# Convert results to a DataFrame
outlier_summary = pd.DataFrame(outlier_results)

# Sort from highest to lowest number of potential outliers
outlier_summary = outlier_summary.sort_values(
    by='Outlier_Count',
    ascending=False
)

print("=== Potential Outlier Summary ===")
print(outlier_summary.to_string(index=False))

=== Potential Outlier Summary ===
                Feature           Q1           Q3          IQR   Lower_Bound  Upper_Bound  Outlier_Count  Outlier_Percentage
        Missed_Payments       0.0000       1.0000       1.0000 -1.500000e+00 2.500000e+00           2739              13.695
           Other_Income       0.0000   16167.2500   16167.2500 -2.425088e+04 4.041812e+04           2737              13.685
          Loan_to_Value      61.8775      70.0400       8.1625  4.963375e+01 8.228375e+01           2574              12.870
          Loan_Defaults       0.0000       0.0000       0.0000  0.000000e+00 0.000000e+00           2484              12.420
       Collateral_Value 2300605.7500 4098911.2500 1798305.5000 -3.968525e+05 6.796370e+06           2263              11.315
            Investments       0.0000  275188.5000  275188.5000 -4.127828e+05 6.879712e+05           1749               8.745
           Credit_Score     572.0000     670.0000      98.0000  4.250000e+02 8.170000e+02  

### Feature Outlier Counts Based on output from above cell

| Feature | Outlier Count |
| :--- | :---: |
| **Missed_Payments** | 2,739 |
| **Other_Income** | 2,737 |
| **Loan_to_Value** | 2,574 |
| **Loan_Defaults** | 2,484 |
| **Collateral_Value** | 2,263 |
| **Investments** | 1,749 |
| **Credit_Score** | 1,137 |
| **Existing_Loans** | 992 |
| **Bank_Balance** | 955 |
| **Years_at_Current_Job** | 924 |
| **Number_of_Bank_Accounts** | 748 |
| **Loan_Amount** | 713 |
| **Monthly_EMI** | 650 |
| **Savings** | 619 |
| **Existing_Loan_Amount** | 613 |
| **Total_Work_Experience** | 588 |
| **Number_of_Credit_Cards** | 548 |
| **Credit_Card_Utilization** | 466 |
| **Monthly_Income** | 358 |
| **Age** | 0 |
| **Dependents** | 0 |
| **City_Tier** | 0 |
| **Debt_to_Income** | 0 |
| **Interest_Rate** | 0 |
| **Loan_Tenure** | 0 |

# Outlier Treatment Classification Framework

> **Guiding Question:** *Does an extreme value represent a data error, or can it legitimately occur in real life?*

---

## 1. Classification Categories

* **⚪ Exclude from IQR (Count / Discrete Variables):** IQR is unsuitable for discrete or low-variance counts (often where $Q_1 = Q_3 = 0$).
* **🟢 Exclude from Removal (Legitimate Extreme Values):** Financial assets and income naturally follow long-tailed distributions; high values reflect real-world wealth, not data errors.
* **🟡 Investigate (Bounded / Logical Ranges):** Variables with hard physical, human, or mathematical limits that require domain validation.
* **🚫 Exclude (Categorical):** Numerically encoded ordinal categories where statistical distance has no mathematical meaning.
* **🔵 No Outliers Detected:** Zero outliers detected using standard IQR thresholds; no filtering needed.

---

## 2. Feature Classification Table

| Feature | Outlier Count | Classification | Primary Rationale |
| :--- | :---: | :---: | :--- |
| **Missed_Payments** | 2,739 | ⚪ Exclude | Discrete count; zero-inflated ($Q_1 = Q_3 = 0$). |
| **Other_Income** | 2,737 | 🟢 Exclude | Wealth variation is expected; do not trim top earners. |
| **Loan_to_Value (LTV)** | 2,574 | 🟡 Investigate | Financial ratio; extreme values impact collateral risk. |
| **Loan_Defaults** | 2,484 | ⚪ Exclude | Discrete count; zero-inflated ($Q_1 = Q_3 = 0$). |
| **Collateral_Value** | 2,263 | 🟢 Exclude | Property values naturally vary across huge ranges. |
| **Investments** | 1,749 | 🟢 Exclude | High investment balances are valid assets. |
| **Credit_Score** | 1,137 | 🟡 Investigate | Must fall within fixed bureau bounds (e.g., $300–850$). |
| **Existing_Loans** | 992 | ⚪ Exclude | Discrete count; multiple loans are not inherently anomalous. |
| **Bank_Balance** | 955 | 🟢 Exclude | High liquidity is legitimate. |
| **Years_at_Current_Job** | 924 | 🟡 Investigate | Cap at total work experience / age plausibility. |
| **Number_of_Bank_Accounts** | 748 | ⚪ Exclude | Discrete count; 4–5 accounts is normal variation. |
| **Loan_Amount** | 713 | 🟢 Exclude | Large requests are valid business cases. |
| **Monthly_EMI** | 650 | 🟢 Exclude | Scales naturally with loan size, tenure, and rate. |
| **Savings** | 619 | 🟢 Exclude | Wealth accumulation varies dramatically. |
| **Existing_Loan_Amount** | 613 | 🟢 Exclude | High debt burdens legitimately exist. |
| **Total_Work_Experience** | 588 | 🟡 Investigate | Check against age and realistic career duration. |
| **Number_of_Credit_Cards** | 548 | ⚪ Exclude | Discrete count; multiple cards are valid. |
| **Credit_Card_Utilization** | 466 | 🟡 Investigate | Percentage bounded ratio; check for values $> 100\%$. |
| **Monthly_Income** | 358 | 🟢 Exclude | High earners are legitimate; long-tailed distribution. |
| **Age** | 0 | 🔵 No Outliers | All values fall within standard IQR bounds ($Q_1 - 1.5\text{IQR}$ to $Q_3 + 1.5\text{IQR}$). |
| **Dependents** | 0 | 🔵 No Outliers | Low variance; all observed counts fall within standard boundaries. |
| **City_Tier** | 0 | 🔵 No Outliers / 🚫 Exclude | Ordinal categorical encoding; 0 outliers detected and math operations are invalid. |
| **Debt_to_Income (DTI)** | 0 | 🔵 No Outliers | Financial ratio; no extreme mathematical anomalies found in current data. |
| **Interest_Rate** | 0 | 🔵 No Outliers | All rate values reside within standard expected distributions. |
| **Loan_Tenure** | 0 | 🔵 No Outliers | Discrete time periods; all values conform to standard allowed loan terms. |

---

## 3. Pipeline Implementation Rules

1. **Never auto-drop 🟢 Green features:** Doing so skews model learning on high-value customers and high-volume transactions.
2. **Apply domain boundary checks on 🟡 Yellow features:** Replace IQR with logical constraints (e.g., $\text{Age} \le 100$, $\text{Utilization} \le 100\%$).
3. **Bypass ⚪ White and 🚫 Categorical features:** Handle via frequency encoding or tree-based splits instead of distance-based outlier filters.
4. **Pass-through 🔵 Blue features:** No action required during preprocessing as these features exhibit no statistical outliers.

### Domain Boundary Validation
  Checking whether the values in the dataset fall within the expected or realistic range for a particular field based on real-world rules and physical constraints.

### Features Requiring Investigation 
***Strategy: Domain Boundary Validation***

| Feature | Outlier Count | Classification | Primary Rationale & Domain Rules |
| :--- | :---: | :---: | :--- |
| **Loan_to_Value (LTV)** | 2,574 | 🟡 Investigate | Financial ratio. Validate against collateral risk bounds (e.g., flag or inspect LTV > 100% or LTV ≤ 0%). |
| **Credit_Score** | 1,137 | 🟡 Investigate | Credit bureau score. Validate against standard range limits (e.g., must strictly fall within 300–850). |
| **Years_at_Current_Job** | 924 | 🟡 Investigate | Human employment limit. Cross-check against `Total_Work_Experience` and overall `Age` plausibility. |
| **Total_Work_Experience** | 588 | 🟡 Investigate | Career duration. Cross-check against `Age` (e.g., rule out $\text{Experience} > [\text{Age} - 16]$). |
| **Credit_Card_Utilization** | 466 | 🟡 Investigate | Bounded percentage ratio. Check for impossible negative values or flag extreme utilization over 100%. |

## Investigation 1: Domain Boundary Validation for Loan-to-Value (LTV)

### 💡 Critical Pause: Data Lineage & Formula Verification

Before jumping straight into **Domain Boundary Validation** for `Loan_to_Value` (LTV), I paused to ask an essential question: 

> *"Was `Loan_to_Value` actually calculated using its standard mathematical formula ($\frac{\text{Loan\_Amount}}{\text{Collateral\_Value}} \times 100$), or is there an underlying discrepancy in the data?"*

To verify this, I decided to run a quick manual calculation across sample rows to build a comparison table between the dataset's existing `Loan_to_Value` column and the formula output:

In [18]:
# ============================================================
# VERIFYING HOW Loan_to_Value WAS CALCULATED
# ============================================================

# Step 1: Select the columns we need
print("Step 1: Checking the relevant columns...")
print()

print(df[[
    "Loan_to_Value",
    "Loan_Amount",
    "Collateral_Value"
]].head(10))


# ============================================================
# Step 2: Calculate LTV ourselves using the assumed formula
# ============================================================

print("\nStep 2: Calculating LTV using:")
print("LTV = (Loan_Amount / Collateral_Value) * 100")
print()

calculated_ltv = (
    df["Loan_Amount"] / df["Collateral_Value"]
) * 100


# ============================================================
# Step 3: Create a comparison table
# ============================================================

comparison = df[[
    "Loan_to_Value",
    "Loan_Amount",
    "Collateral_Value"
]].copy()

comparison["Calculated_LTV"] = calculated_ltv


print("Step 3: Comparing the dataset's LTV with our calculated LTV...")
print()

print(comparison.head(10))



Step 1: Checking the relevant columns...

   Loan_to_Value  Loan_Amount  Collateral_Value
0          82.50       957295         1160384.0
1          62.30      3245374         5209406.0
2          81.11     13847512        17072665.0
3          65.78       277514         3151921.0
4          74.83      1899085         2537834.0
5          58.42      1394906         2387854.0
6          65.78       302340         3151921.0
7          65.78       636773         3151921.0
8          65.78      1233098         3151921.0
9          85.10      2717308         3193149.0

Step 2: Calculating LTV using:
LTV = (Loan_Amount / Collateral_Value) * 100

Step 3: Comparing the dataset's LTV with our calculated LTV...

   Loan_to_Value  Loan_Amount  Collateral_Value  Calculated_LTV
0          82.50       957295         1160384.0       82.498121
1          62.30      3245374         5209406.0       62.298350
2          81.11     13847512        17072665.0       81.109259
3          65.78       277514   

#### 🔍 Findings & Key Takeaway

* **The Observation:** While many rows matched my calculated values perfectly, some diverged. 
* **The Explanation:** This discrepancy was **not a data error**. Because about 37.38% of the missing values in this dataset had been previously imputed using median strategies, downstream mathematical recalculations on those imputed rows naturally differed from original raw ratios. 

#### 🧠 Personal Reflection
This step validated that the column structure was sound and ready for boundary checking, but more importantly, it reinforced an essential truth: **human judgment and critical thinking remain indispensable in the AI era**. 

While AI coding agents can execute scripts instantly, they won't automatically question data lineage or account for prior preprocessing choices like imputation. Your domain intuition as a data practitioner is just as important as the code itself.

### DOMAIN BOUNDARY VALIDATION FOR LOAN-TO-VALUE (LTV)

In [19]:


print("Checking Loan_to_Value domain boundaries...")
print()

# Find LTV values outside the expected 0–100% range
ltv_invalid = df.loc[
    (df["Loan_to_Value"] <= 0) |
    (df["Loan_to_Value"] > 100),
    [
        "Loan_to_Value",
        "Loan_Amount",
        "Collateral_Value"
    ]
]

# Display the invalid/suspicious records
print("Potentially invalid LTV records:")
print(ltv_invalid)

# Count the number of violations
print("\nNumber of LTV domain violations:")
print(len(ltv_invalid))

Checking Loan_to_Value domain boundaries...

Potentially invalid LTV records:
Empty DataFrame
Columns: [Loan_to_Value, Loan_Amount, Collateral_Value]
Index: []

Number of LTV domain violations:
0


### 📊 Domain Boundary Validation Results: `Loan_to_Value`

#### 🔍 Summary of Findings
* **Target Boundary Range:** $0\% < \text{LTV} \le 100\%$
* **Violations Found:** `0` records out of total dataset rows.

#### 💡 Interpretation & Action
The verification confirms that **all `Loan_to_Value` figures strictly adhere to logical financial limits**. There are no negative values, zero-value anomalies, or over-collateralization edge cases exceeding 100% in the dataset. 

Since no operational errors or impossible ratios were detected, **no row deletion or clipping is required** for `Loan_to_Value`. The feature is clean, domain-compliant, and ready for model training.

## Investigation 2: Credit Score

### Strategy & Validation Framework
`Credit_Score` is a strictly bounded financial metric governed by standard credit bureau scoring models (such as FICO or VantageScore). 

* **Valid Domain Bounds:** $[300, 850]$
* **Validation Strategy:**: **Domain Boundary Validation**. 
* **Objective:** Verify that every recorded score falls within $300$ and $850$. Any value below $300$ or above $850$ represents a corrupt data entry or operational error that must be handled.

In [20]:
print("Credit Score Range Check")
print("------------------------")

print("Minimum Credit Score:", df["Credit_Score"].min())
print("Maximum Credit Score:", df["Credit_Score"].max())

# Check values outside the valid range
invalid_scores = df[
    (df["Credit_Score"] < 300) |
    (df["Credit_Score"] > 850)
]

print("\nNumber of invalid Credit Scores:", len(invalid_scores))

print("\nInvalid Credit Scores:")
print(invalid_scores["Credit_Score"])

Credit Score Range Check
------------------------
Minimum Credit Score: 300.0
Maximum Credit Score: 900.0

Number of invalid Credit Scores: 11

Invalid Credit Scores:
2775     897.0
3297     900.0
4334     900.0
6258     860.0
7423     900.0
7656     851.0
11510    900.0
13147    896.0
14610    900.0
16633    866.0
19542    852.0
Name: Credit_Score, dtype: float64


### 📊 Findings & Remediation Strategy

#### Summary of Results
* **Observed Minimum Score:** `300.0` *(Valid lower bound)*
* **Observed Maximum Score:** `900.0` *(Exceeds valid upper bound)*
* **Invalid Records Identified:** **11 records** strictly exceed the $850$ credit bureau threshold (ranging from $851.0$ to $900.0$).

---

#### 🔍 Analysis & Explanation
The minimum score ($300.0$) complies perfectly with domain limits, but **11 observations violate the maximum ceiling of $850$**. 

These values represent clear **data entry errors or system artifacts** (possibly capped at 900 due to a non-standard scoring scale used upstream). Because these 11 rows violate hard real-world constraints, they cannot be accepted as valid credit scores in their current form.

---

#### 🛠️ Recommended Action
Since $11$ records represent a tiny fraction ($<0.1\%$) of the dataset, we have can just to this: 

 **Clip / Cap at Max Bound (Preferred):** Cap these 11 values at the standard maximum limit of `850.0` (`df['Credit_Score'] = df['Credit_Score'].clip(upper=850.0)`). This preserves the underlying applicant profile while bringing the feature into valid domain compliance.




In [21]:
# Cap Credit Score values above 850 at 850
df.loc[df["Credit_Score"] > 850, "Credit_Score"] = 850
print("Maximum Credit Score after correction:", df["Credit_Score"].max())

print("\nNumber of Credit Scores above 850:")
print((df["Credit_Score"] > 850).sum())

Maximum Credit Score after correction: 850.0

Number of Credit Scores above 850:
0


## Investigation 3: Years at Current Job


### Strategy & Validation Framework
`Years_at_Current_Job` is a tenure metric bounded by human employment plausibility. While statistical IQR rules flagged $924$ extreme values for this feature, IQR fails to consider real-world context—a long tenure at a single company is legitimate behavior, not automatically an error.

* **Valid Domain Bounds:** $[0, 50]$ years (based on standard working-age caps).
* **Validation Strategy:** Apply **Domain Boundary Validation** to inspect the minimum and maximum observed values to ensure they fall within realistic human limits.
* **Objective:** Verify that no impossible negative values or non-human upper limits (e.g., $70+$ years at a single company) exist in the data.

In [22]:
print("Minimum years at current job:", df["Years_at_Current_Job"].min())
print("Maximum years at cuurent job:", df["Years_at_Current_Job"].max())

Minimum years at current job: 0
Maximum years at cuurent job: 42


### 📊 Findings & Conclusion

#### Summary of Results
* **Observed Minimum Tenure:** `0` years *(Valid lower bound for new hires)*
* **Observed Maximum Tenure:** `42` years *(Plausible upper bound for long-tenured employees)*
* **Domain Violations Identified:** **0 records**

---

#### 💡 Interpretation
Even though statistical IQR flagged $924$ values as "outliers," domain validation confirms that the maximum tenure of **$42$ years** is entirely realistic for a senior worker or late-career applicant. There are no negative numbers or impossible career spans.

Because all values strictly abide by real-world human bounds, **no rows need to be removed, clipped, or altered**. The statistical "outliers" are legitimate extreme observations that reflect true tenure diversity.

## Investigation 4: Total Work Experience 

### 🔍 Investigation 4: Total Work Experience

### Strategy & Validation Framework
`Total_Work_Experience` measures an applicant's cumulative career duration. Like job tenure, statistical IQR rules flagged $588$ potential "outliers" here, but high work experience naturally occurs among mid- to late-career professionals.

* **Valid Domain Bounds:** $[0, 50]$ years (based on standard career lengths starting from working age $\sim18–25$ up to retirement $\sim65–70$).
* **Validation Strategy:** Apply **Domain Boundary Validation** to verify that career lengths fall within plausible human working-life limits.
* **Objective:** Ensure there are no negative experience figures or impossible values (e.g., $60+$ years of experience) that indicate data corruption.

In [23]:
print("Minimum total work experience:", df["Total_Work_Experience"].min())
print("Maximum total work experience:", df["Total_Work_Experience"].max())
 

Minimum total work experience: 0
Maximum total work experience: 44


### 📊 Findings & Conclusion

#### Summary of Results
* **Observed Minimum Experience:** `0` years *(Valid lower bound for entry-level applicants)*
* **Observed Maximum Experience:** `44` years *(Plausible upper bound for late-career professionals)*
* **Domain Violations Identified:** **0 records**

---

#### 💡 Interpretation
The maximum observed experience of **$44$ years** fits perfectly within a standard $40\text{–}45$ year working lifespan (e.g., an applicant who entered the workforce at age $20$ and is now $64$). 

Although IQR flagged $588$ observations as statistical outliers due to the right-skewed nature of career experience, domain validation proves that **all values are physically and logically realistic**. No rows need to be filtered, clipped, or modified.

## 🔗 Joint Relational Check: `Years_at_Current_Job` vs. `Total_Work_Experience`

### Strategy & Validation Framework
Checking features in isolation is not enough. Individual values like $10$ years at current job and $5$ years total experience look fine on their own, but when paired together, they create a mathematical impossibility.

* **Target Features:** `Years_at_Current_Job` AND `Total_Work_Experience` (Joint Evaluation)
* **Validation Constraint:** $\text{Years\_at\_Current\_Job} \le \text{Total\_Work\_Experience}$
* **Objective:** Verify cross-feature harmony. If a violation occurs, it signals a data entry error in **at least one** of these two interconnected columns.

In [24]:
print("Checking relational consistency...")
print()

relational_mismatch = df.loc[
    df["Years_at_Current_Job"] > df["Total_Work_Experience"],
    [
        "Years_at_Current_Job",
        "Total_Work_Experience"
    ]
]

print("Relational mismatches:")
print(relational_mismatch)

print("\nNumber of relational violations:")
print(len(relational_mismatch))

Checking relational consistency...

Relational mismatches:
Empty DataFrame
Columns: [Years_at_Current_Job, Total_Work_Experience]
Index: []

Number of relational violations:
0


### 📊 Findings & Conclusion

#### Summary of Results
* **Joint Constraint Tested:** $\text{Years\_at\_Current\_Job} \le \text{Total\_Work\_Experience}$
* **Cross-Feature Violations:** **0 records**

---

#### 💡 Interpretation
Zero violations were detected across the entire dataset. This confirms that both employment metrics are mutually consistent—no applicant has a current tenure that exceeds their total career duration. 

This joint validation confirms that both `Years_at_Current_Job` and `Total_Work_Experience` are not only individually realistic, but also **fully aligned with each other**. Both features are validated and ready for model training without modification.

  ## Investigation 5: Credit_Card_Utilization

### Strategy & Validation Framework
`Credit_Card_Utilization` represents the ratio of an applicant's used credit relative to their total limit. While statistical IQR rules flagged $466$ observations as potential outliers, percentage ratios are governed by standard financial boundaries.

* **Valid Domain Bounds:** $[0\%, 100\%]$
* **Validation Strategy:** Enforce **Domain Boundary Validation** to inspect for impossible negative utilization ratios or severe over-limit values.
* **Objective:** Ensure no corrupt data entries ($< 0\%$) or system artifacts ($> 100\%$) exist in the feature before feeding it to downstream models.

In [25]:
print("Minimum credit card utilization:", df["Credit_Card_Utilization"].min())
print("Maximum credit card utilization:", df["Credit_Card_Utilization"].max())
 

Minimum credit card utilization: 0.0
Maximum credit card utilization: 98.98784180197512


### 📊 Findings & Conclusion

#### Summary of Results
* **Observed Minimum Utilization:** `0.0%` *(Valid lower bound)*
* **Observed Maximum Utilization:** `98.99%` *(Valid upper bound)*
* **Domain Violations Identified:** **0 records**

---

#### 💡 Interpretation
The observed minimum of `0.0%` represents applicants using none of their available credit, while the maximum of `98.99%` represents maxed-out borrowers operating just under their credit limits. 

Although statistical IQR rules flagged $466$ observations as outliers due to right-skewness, domain validation proves that **100% of the records sit strictly within the valid financial range ($0\% \le \text{Utilization} \le 100\%$)**. There are no negative balance anomalies or impossible over-limit ratios. 

High-utilization borrowers provide a crucial risk signal for credit decisioning. Because all values represent legitimate financial behavior within valid domain bounds, **no row deletion, clipping, or transformation is required**.

## ⚙️ Encoding Categorical Variables

### ⚙️ Strategy & Method Selection

Machine learning algorithms require numerical inputs to process features effectively. To convert our categorical features into numerical representations without introducing artificial bias, we select the encoding technique based on feature type:

---

#### 1. Binary Encoding (Target & Two-State Features)

Binary encoding is applied when a categorical variable contains exactly two unique classes:

* **Application:** Used primarily for binary features or target variables (e.g., mapping `Loan_Status` as `Approved → 1` and `Rejected → 0`).
* **Conceptual Meaning:** The assigned integers ($0$ and $1$) serve strictly as binary indicators to represent two distinct classes, rather than implying an ordinal hierarchy or rank.

---

#### 2. Ordinal Encoding (Ordered Categories)

Ordinal encoding is applied when categorical variables possess a clear, inherent hierarchy or sequence:

* **Application:** Features with a natural progression (e.g., mapping `Education` as `Primary → 0`, `Secondary → 1`, `Tertiary → 2`, `Postgraduate → 3`).
* **Conceptual Meaning:** Unlike binary encoding, the assigned integers explicitly preserve order ($0 < 1 < 2 < 3$). This communicates to the model that a higher integer corresponds to a higher level within that feature.

---

#### 3. Label Encoding vs. One-Hot Encoding (Nominal Categories)

When dealing with nominal features (categories without a natural order, such as `Marital_Status` or `Employment_Type`), choosing the right transformation is critical:

* **Label Encoding:** Converts nominal categories into arbitrary integers ($0, 1, 2, \dots$). While computationally compact, models cannot inherently tell that these numbers are arbitrary labels. Algorithms may misinterpret them as sequential ($0 < 1 < 2$), introducing artificial bias.
* **One-Hot Encoding (OHE):** Converts nominal categories into separate binary indicator columns ($0$ or $1$). OHE is preferred for nominal features because it completely eliminates unintended ordinal assumptions.

> **Key Takeaway:** Ordinal encoding uses integers to preserve a true category order, whereas One-Hot Encoding is used for unordered categories to prevent algorithms from assuming an artificial rank.

---

#### 4. Statistical Consideration: Multicollinearity Control

* **Dummy Variable Trap:** Multicollinearity control is not an encoding technique itself, but an essential adjustment when using One-Hot Encoding. To prevent perfect linear dependency among generated binary features, we drop one reference category (`drop='first'`).

---

#### 5. Encoding Mapping Plan

We define explicit numerical mappings for binary and ordinal features, and apply One-Hot Encoding with reference dropping to nominal features prior to model training.

### 🔎 Step 1: Identifying Categorical Features

Before selecting an encoding strategy, we must audit the dataset to isolate all object and categorical columns.

* **Objective:** Automatically detect and list all categorical features currently stored as strings or categorical data types.
* **Purpose:** This inspection confirms which columns require numerical conversion before model training and helps us distinguish **ordinal** features (which have an inherent rank) from **nominal** features (which require one-hot encoding).

In [26]:
categorical_columns = df.select_dtypes(
    include=["str", "category"]
).columns

print("Categorical columns:")
print(categorical_columns)

print("\nNumber of categorical columns:", len(categorical_columns))

Categorical columns:
Index(['Marital_Status', 'Education', 'Residence_Type', 'Employment_Type',
       'Tax_Return_Filed', 'Loan_Purpose', 'Collateral', 'Loan_Status'],
      dtype='str')

Number of categorical columns: 8


### 📊 Inspection Results & Encoding Strategy

#### Summary of Identified Features
* **Total Categorical Columns Detected:** `8`
* **Features Isolated:** `Marital_Status`, `Education`, `Residence_Type`, `Employment_Type`, `Tax_Return_Filed`, `Loan_Purpose`, `Collateral`, and `Loan_Status`.

---



### 🔎 Step 2: Inspecting Feature Cardinality

Before applying any transformations, we inspect the number of unique values in each categorical column to choose the correct encoding strategy (e.g., binary mapping for $2$ unique values, ordinal encoding for ordered categories, or one-hot encoding for multi-category nominal features).

In [27]:
for col in categorical_columns:
    print(f"{col}: {df[col].nunique()} unique values")

Marital_Status: 4 unique values
Education: 6 unique values
Residence_Type: 2 unique values
Employment_Type: 5 unique values
Tax_Return_Filed: 3 unique values
Loan_Purpose: 6 unique values
Collateral: 2 unique values
Loan_Status: 2 unique values


#### 💡 Categorical Breakdown & Strategy

Based on the inspection of unique value counts across the 8 categorical features, we categorize and assign encoding techniques as follows:

1. **Target Variable (`Loan_Status` — 2 Unique Values):**
   * **Strategy:** Binary Encoding (`Approved → 1`, `Rejected → 0`).
   * **Rationale:** A standard two-class target variable requiring binary numerical labels for classification models.

2. **Binary / Two-State Features (`Residence_Type` — 2 Unique Values & `Collateral` — 2 Unique Values):**
   * **Strategy:** Binary Indicator / Mapping ($0/1$).
   * **Rationale:** Features with exactly two states do not require multi-column one-hot expansion; mapping directly to $0$ and $1$ keeps the dataset compact without introducing artificial rank.

3. **Ordinal Variable (`Education` — 6 Unique Values):**
   * **Strategy:** Ordinal Integer Mapping ($0, 1, 2, 3, 4, 5$).
   * **Rationale:** Contains 6 distinct tiers with an inherent hierarchical progression (e.g., *High School < Associate < Bachelor's < Master's < Doctorate < Post-Doc*). Sequential integers preserve this logical hierarchy.

4. **Multi-Category Nominal Variables (`Marital_Status` — 4 Unique Values, `Employment_Type` — 5 Unique Values, `Loan_Purpose` — 6 Unique Values):**
   * **Strategy:** One-Hot Encoding (`drop='first'`).
   * **Rationale:** These features possess multiple categories ($4$, $5$, and $6$ respectively) with no inherent ordering or rank. OHE creates binary dummy columns while dropping one reference category to prevent the dummy variable trap (multicollinearity).

5. **Flag Feature (`Tax_Return_Filed` — 3 Unique Values):**
   * **Strategy:** One-Hot Encoding (`drop='first'`) or Categorical Cleaning.
   * **Rationale:** Contains 3 unique values (e.g., `Yes`, `No`, and a potential `Missing`/`Unfiled` indicator). Treating it as a nominal feature ensures each state is captured without assuming sequential order.

### 🔎 Step 3: Inspecting Categorical Values & Cardinality

To select the precise encoding technique for each categorical column, we inspect both the total count of unique values and the specific category labels within every feature.

* **Objective:** Determine feature cardinality and identify underlying domain relationships (binary, ordinal, or nominal).


In [28]:
# Display number of categories and their names

for col in categorical_columns:
    print(f"\n===== {col} =====")
    print("Number of unique values:", df[col].nunique())
    print("Categories:", df[col].unique())


===== Marital_Status =====
Number of unique values: 4
Categories: <StringArray>
['Single', 'Married', 'Widowed', 'Divorced']
Length: 4, dtype: str

===== Education =====
Number of unique values: 6
Categories: <StringArray>
['Graduate', 'Post Graduate', 'PhD', 'High School', 'Diploma', 'No Formal']
Length: 6, dtype: str

===== Residence_Type =====
Number of unique values: 2
Categories: <StringArray>
['Rural', 'Urban']
Length: 2, dtype: str

===== Employment_Type =====
Number of unique values: 5
Categories: <StringArray>
['Private', 'Government', 'Self-Employed', 'Unemployed', 'Skilled Labor']
Length: 5, dtype: str

===== Tax_Return_Filed =====
Number of unique values: 3
Categories: <StringArray>
['Yes', 'No', 'Unknown']
Length: 3, dtype: str

===== Loan_Purpose =====
Number of unique values: 6
Categories: <StringArray>
['Vehicle', 'Home', 'Medical', 'Education', 'Business', 'Personal']
Length: 6, dtype: str

===== Collateral =====
Number of unique values: 2
Categories: <StringArray>
['

### 📊 Category Inspection Results & Encoding Plan

Based on the observed categories across all $8$ non-numeric features, we define the following transformations:

#### 1. Binary Features (Direct 0/1 Mapping)
* **`Loan_Status` (Target):** `{'Rejected': 0, 'Approved': 1}`
* **`Residence_Type`:** `{'Rural': 0, 'Urban': 1}`
* **`Collateral`:** `{'No': 0, 'Yes': 1}`

---

#### 2. Ordinal Feature (Explicit Progression Mapping)
* **`Education`:** Mapped sequentially based on formal attainment level:
  `{'No Formal': 0, 'High School': 1, 'Diploma': 2, 'Graduate': 3, 'Post Graduate': 4, 'PhD': 5}`

---

#### 3. Nominal Features (One-Hot Encoding with `drop='first'`)
* **`Marital_Status` ($4$ categories):** `Single`, `Married`, `Widowed`, `Divorced`
* **`Employment_Type` ($5$ categories):** `Private`, `Government`, `Self-Employed`, `Unemployed`, `Skilled Labor`
* **`Tax_Return_Filed` ($3$ categories):** `Yes`, `No`, `Unknown`
* **`Loan_Purpose` ($6$ categories):** `Vehicle`, `Home`, `Medical`, `Education`, `Business`, `Personal`

> **Note on `Tax_Return_Filed`:** The presence of `'Unknown'` confirms a 3-state nominal feature rather than a simple boolean flag, making One-Hot Encoding the mathematically sound choice.

In [29]:
# ==========================================
# CATEGORICAL VARIABLE ENCODING
# ==========================================

print("==========================================")
print("STARTING CATEGORICAL ENCODING")
print("==========================================")


# ------------------------------------------
# 1. LOAN STATUS — TARGET VARIABLE
# ------------------------------------------

print("\n1. Encoding Loan_Status")
print("Original values:")
print(df["Loan_Status"].value_counts())

df["Loan_Status"] = df["Loan_Status"].map({
    "Rejected": 0,
    "Approved": 1
})

print("\nEncoded values:")
print(df["Loan_Status"].value_counts())

print("Mapping:")
print("Rejected = 0")
print("Approved = 1")


# ------------------------------------------
# 2. RESIDENCE TYPE — BINARY
# ------------------------------------------

print("\n2. Encoding Residence_Type")
print("Original values:")
print(df["Residence_Type"].value_counts())

df["Residence_Type"] = df["Residence_Type"].map({
    "Rural": 0,
    "Urban": 1
})

print("\nEncoded values:")
print(df["Residence_Type"].value_counts())

print("Mapping:")
print("Rural = 0")
print("Urban = 1")


# ------------------------------------------
# 3. COLLATERAL — BINARY
# ------------------------------------------

print("\n3. Encoding Collateral")
print("Original values:")
print(df["Collateral"].value_counts())

df["Collateral"] = df["Collateral"].map({
    "No": 0,
    "Yes": 1
})

print("\nEncoded values:")
print(df["Collateral"].value_counts())

print("Mapping:")
print("No = 0")
print("Yes = 1")


# ------------------------------------------
# 4. EDUCATION — ORDINAL
# ------------------------------------------

print("\n4. Encoding Education")
print("Original values:")
print(df["Education"].value_counts())

education_mapping = {
    "No Formal": 0,
    "High School": 1,
    "Diploma": 2,
    "Graduate": 3,
    "Post Graduate": 4,
    "PhD": 5
}

df["Education"] = df["Education"].map(education_mapping)

print("\nEncoded values:")
print(df["Education"].value_counts())

print("\nEducation mapping:")
for category, value in education_mapping.items():
    print(f"{category} = {value}")


# ------------------------------------------
# 5. ONE-HOT ENCODING
# ------------------------------------------

print("\n5. One-Hot Encoding nominal features")

print("Features:")
print("Marital_Status")
print("Employment_Type")
print("Tax_Return_Filed")
print("Loan_Purpose")

df = pd.get_dummies(
    df,
    columns=[
        "Marital_Status",
        "Employment_Type",
        "Tax_Return_Filed",
        "Loan_Purpose"
    ],
    drop_first=True,
    dtype=int
)

print("\nOne-Hot Encoding completed.")


# ------------------------------------------
# 6. CHECK REMAINING CATEGORICAL COLUMNS
# ------------------------------------------

print("\n==========================================")
print("CHECKING REMAINING CATEGORICAL COLUMNS")
print("==========================================")

remaining_categorical = df.select_dtypes(
    include=["str", "category"]
).columns

print("\nRemaining categorical columns:")
print(remaining_categorical)

print("\nNumber of remaining categorical columns:")
print(len(remaining_categorical))


# ------------------------------------------
# 7. CHECK DATA TYPES
# ------------------------------------------


STARTING CATEGORICAL ENCODING

1. Encoding Loan_Status
Original values:
Loan_Status
Approved    12856
Rejected     7144
Name: count, dtype: int64

Encoded values:
Loan_Status
1    12856
0     7144
Name: count, dtype: int64
Mapping:
Rejected = 0
Approved = 1

2. Encoding Residence_Type
Original values:
Residence_Type
Urban    14972
Rural     5028
Name: count, dtype: int64

Encoded values:
Residence_Type
1    14972
0     5028
Name: count, dtype: int64
Mapping:
Rural = 0
Urban = 1

3. Encoding Collateral
Original values:
Collateral
Yes    12605
No      7395
Name: count, dtype: int64

Encoded values:
Collateral
1    12605
0     7395
Name: count, dtype: int64
Mapping:
No = 0
Yes = 1

4. Encoding Education
Original values:
Education
Graduate         6978
High School      5034
Post Graduate    3283
Diploma          2403
No Formal        1644
PhD               658
Name: count, dtype: int64

Encoded values:
Education
3    6978
1    5034
4    3283
2    2403
0    1644
5     658
Name: count, dtype

### VERIFICATION OF CATEGORICAL ENCODING

In [30]:

print("\n==========================================")
print("ENCODED COLUMNS")
print("==========================================")

print(df.columns.tolist())


# ------------------------------------------
# 1. VERIFY BINARY ENCODING
# ------------------------------------------

print("\n==========================================")
print("BINARY ENCODING VERIFICATION")
print("==========================================")

binary_columns = [
    "Loan_Status",
    "Residence_Type",
    "Collateral"
]

for col in binary_columns:
    print(f"\n{col}:")
    print(df[col].value_counts().sort_index())


# ------------------------------------------
# 2. VERIFY ORDINAL ENCODING
# ------------------------------------------

print("\n==========================================")
print("ORDINAL ENCODING VERIFICATION")
print("==========================================")

print("\nEducation value counts:")
print(df["Education"].value_counts().sort_index())


# ------------------------------------------
# 3. VERIFY ONE-HOT ENCODING
# ------------------------------------------

print("\n==========================================")
print("ONE-HOT ENCODING VERIFICATION")
print("==========================================")

one_hot_columns = [
    col for col in df.columns
    if (
        col.startswith("Marital_Status_")
        or col.startswith("Employment_Type_")
        or col.startswith("Tax_Return_Filed_")
        or col.startswith("Loan_Purpose_")
    )
]

print("\nGenerated one-hot columns:")
for col in one_hot_columns:
    print(col)





ENCODED COLUMNS
['Age', 'Education', 'Dependents', 'Residence_Type', 'City_Tier', 'Years_at_Current_Job', 'Total_Work_Experience', 'Monthly_Income', 'Other_Income', 'Existing_Loans', 'Existing_Loan_Amount', 'Monthly_EMI', 'Debt_to_Income', 'Savings', 'Investments', 'Bank_Balance', 'Credit_Card_Utilization', 'Number_of_Bank_Accounts', 'Number_of_Credit_Cards', 'Credit_Score', 'Loan_Defaults', 'Missed_Payments', 'Loan_Amount', 'Loan_Tenure', 'Interest_Rate', 'Collateral', 'Collateral_Value', 'Loan_to_Value', 'Loan_Status', 'Marital_Status_Married', 'Marital_Status_Single', 'Marital_Status_Widowed', 'Employment_Type_Private', 'Employment_Type_Self-Employed', 'Employment_Type_Skilled Labor', 'Employment_Type_Unemployed', 'Tax_Return_Filed_Unknown', 'Tax_Return_Filed_Yes', 'Loan_Purpose_Education', 'Loan_Purpose_Home', 'Loan_Purpose_Medical', 'Loan_Purpose_Personal', 'Loan_Purpose_Vehicle']

BINARY ENCODING VERIFICATION

Loan_Status:
Loan_Status
0     7144
1    12856
Name: count, dtype: in

## Feature Engineering

Feature engineering is the process of creating new, informative features from existing raw data to help machine learning algorithms uncover underlying patterns more effectively.

For our **Loan Approval Decision Tree**, we adhere to a purposeful feature creation strategy:

* **Domain-Driven Design:** We do not create features arbitrarily or purely to expand dataset width.
* **Logical Alignment:** Every engineered feature must share a clear, explainable relationship with applicant credit risk and loan approval probability.
* **Model Interpretability:** Quality features enhance tree splitting efficiency, leading to simpler, highly interpretable decision nodes.

### 🔎 Auditing Existing Features Prior to Engineering

Before constructing new indicators, we perform a thorough review of our cleaned, baseline feature set.

* **Baseline Inventory:** Reviewing all available numerical and categorical columns currently in the dataset.
* **Domain Relevance:** Identifying key financial metrics—such as income, debt obligations, assets, and loan terms—that can be combined into meaningful risk ratios.
* **Redundancy Avoidance:** Ensuring newly engineered variables directly complement, rather than duplicate, the signal already present in the existing features.

In [31]:
print("=========================================")
print("CURRENT DATASET FEATURES")
print("=========================================")

print(df.columns.tolist())

print("\nTotal number of features:")
print(len(df.columns))

CURRENT DATASET FEATURES
['Age', 'Education', 'Dependents', 'Residence_Type', 'City_Tier', 'Years_at_Current_Job', 'Total_Work_Experience', 'Monthly_Income', 'Other_Income', 'Existing_Loans', 'Existing_Loan_Amount', 'Monthly_EMI', 'Debt_to_Income', 'Savings', 'Investments', 'Bank_Balance', 'Credit_Card_Utilization', 'Number_of_Bank_Accounts', 'Number_of_Credit_Cards', 'Credit_Score', 'Loan_Defaults', 'Missed_Payments', 'Loan_Amount', 'Loan_Tenure', 'Interest_Rate', 'Collateral', 'Collateral_Value', 'Loan_to_Value', 'Loan_Status', 'Marital_Status_Married', 'Marital_Status_Single', 'Marital_Status_Widowed', 'Employment_Type_Private', 'Employment_Type_Self-Employed', 'Employment_Type_Skilled Labor', 'Employment_Type_Unemployed', 'Tax_Return_Filed_Unknown', 'Tax_Return_Filed_Yes', 'Loan_Purpose_Education', 'Loan_Purpose_Home', 'Loan_Purpose_Medical', 'Loan_Purpose_Personal', 'Loan_Purpose_Vehicle']

Total number of features:
43


### 💡 Why the Dataset Increased from 33 to 43 Columns

The dataset initially contained **33 columns** following the data cleaning stage. After applying One-Hot Encoding (OHE) to nominal categorical variables, the total column count expanded to **43 columns**.

It is important to note that this increase does **not** indicate that 10 new domain features were created through Feature Engineering. Instead, four original categorical string columns were transformed into multiple binary ($0/1$) indicator columns.

---

#### Column Expansion Breakdown

| Original Feature | One-Hot Encoded Dummy Columns (`drop='first'`) | Net Column Increase |
| :--- | :--- | :---: |
| `Marital_Status` | `Marital_Status_Married`, `Marital_Status_Single`, `Marital_Status_Widowed` | **+2** |
| `Employment_Type` | `Employment_Type_Private`, `Employment_Type_Self-Employed`, `Employment_Type_Skilled Labor`, `Employment_Type_Unemployed` | **+3** |
| `Tax_Return_Filed` | `Tax_Return_Filed_Unknown`, `Tax_Return_Filed_Yes` | **+1** |
| `Loan_Purpose` | `Loan_Purpose_Education`, `Loan_Purpose_Home`, `Loan_Purpose_Medical`, `Loan_Purpose_Personal`, `Loan_Purpose_Vehicle` | **+4** |
| **Total Net Increase** | | **+10** |

---

#### How the Net Increase Works

When a categorical feature is one-hot encoded with one category dropped (`drop='first'`) to prevent multicollinearity, the original single column is removed and replaced by $N - 1$ binary dummy columns (where $N$ is the number of unique categories):

$$\text{Net Column Increase} = (N - 1) - 1$$

For example, `Marital_Status` originally contained $4$ unique categories:

$$\text{Marital\_Status} \longrightarrow \begin{cases} \text{Marital\_Status\_Married} \\ \text{Marital\_Status\_Single} \\ \text{Marital\_Status\_Widowed} \end{cases}$$

The $1$ original string column was replaced by $3$ binary dummy columns, yielding a **net gain of 2 columns**.

---

#### Final Tally

* **Original Cleaned Columns:** $33$
* **Net Increase from OHE:** $+10$
* **Current Post-Encoding Columns:** $43$

$$33 + 10 = 43 \text{ columns}$$

---

> **Summary:** The shift from 33 to 43 columns is strictly a numerical representation of existing nominal features required for algorithm compatibility. Prior to starting our explicit Feature Engineering step, no new domain indicators have been introduced.

### 🛠️ Feature Engineering: Creating `Total_Debt_Exposure`

To give our model a clearer view of an applicant's total financial liabilities, we derive a consolidated liability metric though it is still possible for the model to figure this out on its own but we want to give the model the relationship directly instead of requiring the model to discover it from the two separate variables.

* **Rationale:** A borrower's risk profile depends on their total financial burden upon approval. Combining existing debt obligations with the requested principal provides a more realistic representation of their overall debt exposure.
* **Formula:**
  $$\text{Total\_Debt\_Exposure} = \text{Existing\_Loan\_Amount} + \text{Loan\_Amount}$$

In [32]:
# =========================================
# FEATURE ENGINEERING
# =========================================

df["Total_Debt_Exposure"] = (
    df["Existing_Loan_Amount"] + df["Loan_Amount"]
)

print("=========================================")
print("FEATURE ENGINEERING")
print("=========================================")

print("New feature created: Total_Debt_Exposure")

print("\nSample calculation:")
print(
    df[
        [
            "Existing_Loan_Amount",
            "Loan_Amount",
            "Total_Debt_Exposure"
        ]
    ].head(10)
)

print("\nNew feature statistics:")
print(df["Total_Debt_Exposure"].describe())

print("\nNew dataset shape:")
print(df.shape)

FEATURE ENGINEERING
New feature created: Total_Debt_Exposure

Sample calculation:
   Existing_Loan_Amount  Loan_Amount  Total_Debt_Exposure
0                     0       957295               957295
1                380423      3245374              3625797
2              12237646     13847512             26085158
3                     0       277514               277514
4               2664833      1899085              4563918
5                     0      1394906              1394906
6                     0       302340               302340
7                     0       636773               636773
8                414907      1233098              1648005
9                     0      2717308              2717308

New feature statistics:
count    2.000000e+04
mean     3.530972e+06
std      3.411675e+06
min      5.322000e+03
25%      1.117213e+06
50%      2.811021e+06
75%      4.961191e+06
max      1.156434e+08
Name: Total_Debt_Exposure, dtype: float64

New dataset shape:
(20000, 44)


### 📊 Feature Engineering Verification & Statistics

The engineered feature `Total_Debt_Exposure` has been successfully computed and added to the dataset:

* **Column Addition:** The dataset width expanded by $+1$ column (increasing total features from $43$ to $44$).
* **Data Integrity:** Sample checks confirm that `Total_Debt_Exposure` accurately reflects the sum of `Existing_Loan_Amount` and `Loan_Amount` across all rows without introducing missing or NaN values.

## 💾 12. Exporting Leakage-Safe Processed Datasets

The earlier exploration above uses `df` to inspect and understand the data. For model-ready exports, we now restart from the raw dataset and use the standard machine-learning flow:

1. Split raw features and labels before fitting preprocessing.
2. Fit each data-dependent operation on `X_train` only.
3. Apply the fitted training rules unchanged to `X_test`.
4. Validate the final numeric feature matrices before saving.

This prevents the test set from influencing imputation values or the one-hot-encoding schema, which would otherwise cause data leakage.

In [ ]:
# ============================================================
# LEAKAGE-SAFE PREPROCESSING AND EXPORT
# ============================================================

# ------------------------------------------------------------
# 1. Reload raw data and apply fixed feature-selection rules
# ------------------------------------------------------------
# Reloading guarantees that the exported data does not reuse the
# earlier exploratory transformations performed on df.
model_df = pd.read_csv("../data/raw/synthetic_loan_approval_dataset.csv")

columns_to_drop = [
    "Application_ID", "Gender", "Annual_Income", "PAN_Verified",
    "Aadhaar_Verified", "Property_Value"
]
model_df = model_df.drop(columns=columns_to_drop)

# Separate raw features and target before any learned preprocessing.
X = model_df.drop(columns="Loan_Status")
y = model_df["Loan_Status"].map({"Rejected": 0, "Approved": 1})

# ------------------------------------------------------------
# 2. Split before fitting preprocessing
# ------------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
X_train = X_train.copy()
X_test = X_test.copy()

# ------------------------------------------------------------
# 3. Fit numerical imputations on X_train only
# ------------------------------------------------------------
# Fixed domain values can be applied directly to both datasets.
for dataset in (X_train, X_test):
    dataset["Other_Income"] = dataset["Other_Income"].fillna(0)
    dataset["Investments"] = dataset["Investments"].fillna(0)
    dataset["Tax_Return_Filed"] = dataset["Tax_Return_Filed"].fillna("Unknown")

# Calculate each learned median from training data only.
median_columns = [
    "Loan_to_Value", "Collateral_Value", "Bank_Balance",
    "Credit_Score", "Interest_Rate"
]
training_medians = X_train[median_columns].median()

for column in median_columns:
    X_train[column] = X_train[column].fillna(training_medians[column])
    X_test[column] = X_test[column].fillna(training_medians[column])

# Apply the fixed credit-score business boundary to both datasets.
for dataset in (X_train, X_test):
    dataset["Credit_Score"] = dataset["Credit_Score"].clip(upper=850)

# ------------------------------------------------------------
# 4. Create deterministic feature engineering in both datasets
# ------------------------------------------------------------
for dataset in (X_train, X_test):
    dataset["Total_Debt_Exposure"] = (
        dataset["Existing_Loan_Amount"] + dataset["Loan_Amount"]
    )

# ------------------------------------------------------------
# 5. Apply fixed binary and ordinal mappings
# ------------------------------------------------------------
education_mapping = {
    "No Formal": 0, "High School": 1, "Diploma": 2,
    "Graduate": 3, "Post Graduate": 4, "PhD": 5
}

for dataset in (X_train, X_test):
    dataset["Residence_Type"] = dataset["Residence_Type"].map({"Rural": 0, "Urban": 1})
    dataset["Collateral"] = dataset["Collateral"].map({"No": 0, "Yes": 1})
    dataset["Education"] = dataset["Education"].map(education_mapping)

# ------------------------------------------------------------
# 6. Fit one-hot encoder on X_train and transform X_test
# ------------------------------------------------------------
nominal_features = [
    "Marital_Status", "Employment_Type", "Tax_Return_Filed", "Loan_Purpose"
]
encoder = OneHotEncoder(
    drop="first", handle_unknown="ignore", sparse_output=False, dtype=int
).set_output(transform="pandas")

encoded_train = encoder.fit_transform(X_train[nominal_features])
encoded_test = encoder.transform(X_test[nominal_features])

X_train = pd.concat([
    X_train.drop(columns=nominal_features).reset_index(drop=True),
    encoded_train.reset_index(drop=True)
], axis=1)
X_test = pd.concat([
    X_test.drop(columns=nominal_features).reset_index(drop=True),
    encoded_test.reset_index(drop=True)
], axis=1)

# ------------------------------------------------------------
# 7. Validate before saving
# ------------------------------------------------------------
assert X_train.columns.equals(X_test.columns), "Train and test schemas differ."
assert X_train.select_dtypes(exclude="number").empty, "X_train has non-numeric columns."
assert X_test.select_dtypes(exclude="number").empty, "X_test has non-numeric columns."
assert not X_train.isna().any().any(), "X_train has missing values."
assert not X_test.isna().any().any(), "X_test has missing values."
assert not y_train.isna().any(), "y_train has missing values."
assert not y_test.isna().any(), "y_test has missing values."

# ------------------------------------------------------------
# 8. Save processed datasets
# ------------------------------------------------------------
X_train.to_csv("../data/processed/X_train_processed.csv", index=False)
X_test.to_csv("../data/processed/X_test_processed.csv", index=False)
y_train.to_frame(name="Loan_Status").to_csv("../data/processed/y_train.csv", index=False)
y_test.to_frame(name="Loan_Status").to_csv("../data/processed/y_test.csv", index=False)

print("=" * 60)
print("LEAKAGE-SAFE PREPROCESSING COMPLETE")
print("=" * 60)
print("\nX_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("Number of features:", X_train.shape[1])
print("Feature schemas match:", X_train.columns.equals(X_test.columns))
print("Missing values in X_train:", X_train.isna().sum().sum())
print("Missing values in X_test:", X_test.isna().sum().sum())
print("\nAll processed datasets saved successfully!")

### ✅ Preprocessing Pipeline Summary

The exported model datasets now follow a leakage-safe standard ML workflow:

* **Raw split first:** The raw feature matrix is split into stratified training and test subsets before learned preprocessing begins.
* **Training-only fitting:** Median values and one-hot category definitions are fit exclusively on `X_train`.
* **Consistent transformation:** The fitted training rules are applied unchanged to `X_test`.
* **Numeric model inputs:** Binary, ordinal, and nominal categorical variables are converted to numeric columns.
* **Export validation:** Both feature files are checked for matching schemas and zero missing values before saving.

> **Pipeline Complete:** `X_train_processed.csv`, `X_test_processed.csv`, `y_train.csv`, and `y_test.csv` are ready for cross-validation and final test evaluation.